In [ ]:
# Verification: Sample ~10 phasors before/after correction
print("\n=== Bode Correction Verification ===")
sample_count = 0
for sensor_idx in sensor_indices[:min(3, len(sensor_indices))]:  # First 3 sensors
    sensor_name = sensor_name_arr.sel({sensor_dim: sensor_idx}).item()
    if isinstance(sensor_name, bytes):
        sensor_name = sensor_name.decode()
    
    # Sample every Nth frequency to avoid too much output
    freq_step = max(1, len(freq_vals) // 4)
    for freq_idx in range(0, len(freq_vals), freq_step)[:4]:  # ~4 freq samples per sensor
        # Get phasor and average/flatten over other dimensions (like time)
        phasor_before = complex_fft.isel({sensor_dim: sensor_idx, freq_dim: freq_idx}).mean().item()
        phasor_after = complex_fft_work.isel({sensor_dim: sensor_idx, freq_dim: freq_idx}).mean().item()
        
        mag_before = np.abs(phasor_before)
        mag_after = np.abs(phasor_after)
        phase_before = np.angle(phasor_before) * 180 / np.pi
        phase_after = np.angle(phasor_after) * 180 / np.pi
        
        print(f"\nSensor {sensor_idx} ({sensor_name}), Freq {freq_vals[freq_idx]:.1e} Hz:")
        print(f"  Before: mag={mag_before:.3e}, phase={phase_before:+.1f}°")
        print(f"  After:  mag={mag_after:.3e}, phase={phase_after:+.1f}°")
        print(f"  Ratio:  {mag_after/max(mag_before, 1e-20):.3f}x")
        
        sample_count += 1
        if sample_count >= 10:
            break
    if sample_count >= 10:
        break
print("=== End Verification ===\n")

In [ ]:
from pathlib import Path
import numpy as npcomplex_fft = ds["mirnov_fft_real"] + 1j * ds["mirnov_fft_imag"]

    sensor_dim = "sensor"
    freq_dim = next((d for d in complex_fft.dims if "freq" in d.lower()), None)
    if freq_dim is None:
        raise KeyError(
            f"Could not find frequency dimension in {complex_fft.dims}. "
            "Expected a dimension name containing 'freq'."
        )

    sensor_name_key = None
    for candidate in ("sensor_names", "sensor_name"):
        if candidate in ds.coords or candidate in ds.variables:
            sensor_name_key = candidate
            break
    if sensor_name_key is None:
        raise KeyError("Could not find sensor name field. Expected 'sensor_names' or 'sensor_name'.")

    other_dims = [d for d in complex_fft.dims if d not in (sensor_dim, freq_dim)]
    corrected = complex_fft.transpose(sensor_dim, freq_dim, *other_dims).copy(deep=True)

    freq_vals = ds.coords[freq_dim].values
    sensor_indices = ds.coords[sensor_dim].values
    sensor_name_arr = ds[sensor_name_key]

    for sensor_idx in sensor_indices:
        sensor_name = sensor_name_arr.sel({sensor_dim: sensor_idx}).item()
        if isinstance(sensor_name, bytes):
            sensor_name = sensor_name.decode()

        _, H = __cal_Correction(str(sensor_name), freq_vals)
        if np.any(np.abs(H) == 0):
            raise ZeroDivisionError(
                f"Calibration transfer function contains zeros for sensor {sensor_name}."
            )

        corrected.loc[{sensor_dim: sensor_idx}] = (
            corrected.sel({sensor_dim: sensor_idx})
            / xr.DataArray(H, dims=[freq_dim], coords={freq_dim: freq_vals})
        )

    corrected = corrected.transpose(*complex_fft.dims)
    ds["mirnov_fft_real"] = corrected.real
    ds["mirnov_fft_imag"] = corrected.imag
import xarray as xr
from header_Cmod import __cal_Correction


def apply_bode_correction_to_file(
    file_path,
    overwrite=True,
    fallback_suffix="_bode_corrected",
):
    """Apply Bode correction to mirnov FFT fields and save dataset.

    Expects dataset fields:
      - mirnov_fft_real, mirnov_fft_imag
      - sensor coordinate
      - sensor name field: sensor_names or sensor_name
      - frequency coordinate dimension name containing 'freq'
    """
    file_path = str(file_path)

    with xr.open_dataset(file_path) as ds_in:
        ds = ds_in.load()

    complex_fft = ds["mirnov_fft_real"] + 1j * ds["mirnov_fft_imag"]

    sensor_dim = "sensor"
    freq_dim = next((d for d in complex_fft.dims if "freq" in d.lower()), None)
    if freq_dim is None:
        raise KeyError(
            f"Could not find frequency dimension in {complex_fft.dims}. "
            "Expected a dimension name containing 'freq'."
        )

    sensor_name_key = None
    for candidate in ("sensor_names", "sensor_name"):
        if candidate in ds.coords or candidate in ds.variables:
            sensor_name_key = candidate
            break
    if sensor_name_key is None:
        raise KeyError("Could not find sensor name field. Expected 'sensor_names' or 'sensor_name'.")

    other_dims = [d for d in complex_fft.dims if d not in (sensor_dim, freq_dim)]
    corrected = complex_fft.transpose(sensor_dim, freq_dim, *other_dims).copy(deep=True)

    freq_vals = ds.coords[freq_dim].values
    sensor_indices = ds.coords[sensor_dim].values
    sensor_name_arr = ds[sensor_name_key]

    for sensor_idx in sensor_indices:
        sensor_name = sensor_name_arr.sel({sensor_dim: sensor_idx}).item()
        if isinstance(sensor_name, bytes):
            sensor_name = sensor_name.decode()

        _, H = __cal_Correction(str(sensor_name), freq_vals)
        if np.any(np.abs(H) == 0):
            raise ZeroDivisionError(
                f"Calibration transfer function contains zeros for sensor {sensor_name}."
            )

        corrected.loc[{sensor_dim: sensor_idx}] = (
            corrected.sel({sensor_dim: sensor_idx})
            / xr.DataArray(H, dims=[freq_dim], coords={freq_dim: freq_vals})
        )

    corrected = corrected.transpose(*complex_fft.dims)
    ds["mirnov_fft_real"] = corrected.real
    ds["mirnov_fft_imag"] = corrected.imag

    target = file_path
    if not overwrite:
        p = Path(file_path)
        target = str(p.with_name(f"{p.stem}{fallback_suffix}{p.suffix}"))

    try:
        ds.to_netcdf(target, mode="w")
    except PermissionError:
        p = Path(file_path)
        target = str(p.with_name(f"{p.stem}{fallback_suffix}{p.suffix}"))
        ds.to_netcdf(target, mode="w")

    return target


def apply_bode_correction_batch(file_paths, overwrite=True, continue_on_error=True):
    """Apply correction to multiple files and return {file_path: output_path_or_error}."""
    results = {}
    for file_path in file_paths:
        try:
            results[str(file_path)] = apply_bode_correction_to_file(
                file_path, overwrite=overwrite
            )
        except Exception as exc:
            results[str(file_path)] = f"ERROR: {exc}"
            if not continue_on_error:
                raise
    return results


# Example usage:
out_path = apply_bode_correction_to_file("/home/rianc/Documents/TARS/tars/scratch/input_data/1160930034.nc", overwrite=False)
# results = apply_bode_correction_batch(["/path/a.nc", "/path/b.nc"], overwrite=False)
print(out_path)
#print(results)